# OffScript — Phase 6: Deployment Data Preparation

This notebook creates a minimal subset of the processed data files
for use in the live deployment. The full processed datasets are
gitignored due to size but a representative subset is committed
to enable the deployed API to serve real data rather than mock data.

**Input:** Full parquet files in `data/processed/`
**Output:** Minimal parquet files in `data/deploy/`

## Imports and Load

In [1]:
import sys
sys.path.append('../src')

import pandas as pd
import numpy as np
import os

# Load all processed datasets
pitcher_profiles = pd.read_parquet('../data/processed/pitcher_profiles.parquet')
matchup_scores = pd.read_parquet('../data/processed/matchup_scores.parquet')
deviation_effectiveness = pd.read_parquet('../data/processed/deviation_effectiveness.parquet')
batter_vulnerability = pd.read_parquet('../data/processed/batter_vulnerability.parquet')
pitcher_data = pd.read_parquet('../data/processed/pitcher_data_with_recommendations.parquet')

print("All datasets loaded successfully")

All datasets loaded successfully


## Check Sizes

In [2]:
datasets = {
    'pitcher_profiles': pitcher_profiles,
    'matchup_scores': matchup_scores,
    'deviation_effectiveness': deviation_effectiveness,
    'batter_vulnerability': batter_vulnerability,
    'pitcher_data': pitcher_data
}

print("=== Dataset Size Report ===\n")
total_kb = 0
for name, df in datasets.items():
    rows = len(df)
    kb = df.memory_usage(deep=True).sum() / 1024
    total_kb += kb
    print(f"{name}:")
    print(f"  Rows:   {rows:,}")
    print(f"  Size:   {kb:.1f} KB ({kb/1024:.2f} MB)")
    print(f"  Cols:   {list(df.columns)}")
    print()

print(f"Total in-memory size: {total_kb:.1f} KB ({total_kb/1024:.2f} MB)")

=== Dataset Size Report ===

pitcher_profiles:
  Rows:   14
  Size:   1.3 KB (0.00 MB)
  Cols:   ['pitcher_name', 'total_pitches', 'deviation_score', 'deviation_positive_rate', 'followed_positive_rate', 'deviation_advantage', 'two_strike_dev_cost', 'arsenal_size', 'deviation_rank']

matchup_scores:
  Rows:   5,786
  Size:   376.0 KB (0.37 MB)
  Cols:   ['pitcher_name', 'batter_id', 'batter_name', 'stand', 'matchup_score']

deviation_effectiveness:
  Rows:   14
  Size:   0.5 KB (0.00 MB)
  Cols:   ['deviation_positive_rate', 'followed_positive_rate', 'deviation_advantage']

batter_vulnerability:
  Rows:   1,148
  Size:   89.8 KB (0.09 MB)
  Cols:   ['batter', 'batter_name', 'stand', 'pitch_type', 'pitches_seen', 'whiff_rate', 'hit_rate', 'vulnerability_score']

pitcher_data:
  Rows:   72,098
  Size:   25252.1 KB (24.66 MB)
  Cols:   ['game_date', 'pitcher', 'player_name', 'pitch_type', 'pitch_name', 'release_speed', 'pfx_x', 'pfx_z', 'plate_x', 'plate_z', 'balls', 'strikes', 'on_1b', 'o

## Create Deploy Directory

Four datasets are small enough to commit directly. The pitcher_data
file is reduced to only the columns required by the API endpoints,
cutting the size dramatically.

### API column requirements
- Arsenal endpoint: `pitcher_name`, `pitch_type`
- Deviations endpoint: `pitcher_name`, `pitch_type`, 
  `recommended_pitch`, `followed_recommendation`

## Create Deploy Directory and Save Small Files

In [3]:
# Create deploy directory
os.makedirs('../data/deploy', exist_ok=True)

# Save small files directly — no reduction needed
pitcher_profiles.to_parquet(
    '../data/deploy/pitcher_profiles.parquet', index=False
)
matchup_scores.to_parquet(
    '../data/deploy/matchup_scores.parquet', index=False
)
deviation_effectiveness.to_parquet(
    '../data/deploy/deviation_effectiveness.parquet', index=False
)
batter_vulnerability.to_parquet(
    '../data/deploy/batter_vulnerability.parquet', index=False
)

print("Small files saved to data/deploy/")
print(f"  pitcher_profiles:      {os.path.getsize('../data/deploy/pitcher_profiles.parquet') / 1024:.1f} KB")
print(f"  matchup_scores:        {os.path.getsize('../data/deploy/matchup_scores.parquet') / 1024:.1f} KB")
print(f"  deviation_effectiveness: {os.path.getsize('../data/deploy/deviation_effectiveness.parquet') / 1024:.1f} KB")
print(f"  batter_vulnerability:  {os.path.getsize('../data/deploy/batter_vulnerability.parquet') / 1024:.1f} KB")

Small files saved to data/deploy/
  pitcher_profiles:      7.0 KB
  matchup_scores:        46.6 KB
  deviation_effectiveness: 2.9 KB
  batter_vulnerability:  24.3 KB


## Slim Pitcher Data

In [4]:
# Keep only columns required by API endpoints
api_cols = [
    'pitcher_name',
    'pitch_type',
    'recommended_pitch',
    'followed_recommendation'
]

pitcher_data_slim = pitcher_data[api_cols].copy()

print(f"Original pitcher_data:  {len(pitcher_data):,} rows, "
      f"{pitcher_data.memory_usage(deep=True).sum() / 1024:.1f} KB")
print(f"Slimmed pitcher_data:   {len(pitcher_data_slim):,} rows, "
      f"{pitcher_data_slim.memory_usage(deep=True).sum() / 1024:.1f} KB")

pitcher_data_slim.to_parquet(
    '../data/deploy/pitcher_data.parquet', index=False
)

actual_size = os.path.getsize('../data/deploy/pitcher_data.parquet') / 1024
print(f"\nSaved to data/deploy/pitcher_data.parquet")
print(f"On-disk size: {actual_size:.1f} KB ({actual_size/1024:.2f} MB)")

Original pitcher_data:  72,098 rows, 25252.1 KB
Slimmed pitcher_data:   72,098 rows, 2927.2 KB

Saved to data/deploy/pitcher_data.parquet
On-disk size: 70.4 KB (0.07 MB)


## Verify Total Deploy Size

In [5]:
print("=== Deploy Directory Size Summary ===\n")
total_deploy_kb = 0

for filename in os.listdir('../data/deploy'):
    if filename.endswith('.parquet'):
        path = f'../data/deploy/{filename}'
        size_kb = os.path.getsize(path) / 1024
        total_deploy_kb += size_kb
        print(f"{filename}: {size_kb:.1f} KB")

print(f"\nTotal deploy size: {total_deploy_kb:.1f} KB "
      f"({total_deploy_kb/1024:.2f} MB)")
print(f"Reduction from full dataset: "
      f"{25719.6 - total_deploy_kb:.1f} KB saved")

=== Deploy Directory Size Summary ===

batter_vulnerability.parquet: 24.3 KB
deviation_effectiveness.parquet: 2.9 KB
matchup_scores.parquet: 46.6 KB
pitcher_data.parquet: 70.4 KB
pitcher_profiles.parquet: 7.0 KB

Total deploy size: 151.2 KB (0.15 MB)
Reduction from full dataset: 25568.4 KB saved


## Model Files

The three model pkl files also need to be available in deployment:
- `models/baseline_pitch_model.pkl`
- `models/label_encoder.pkl`  
- `models/pitcher_encoder.pkl`

These are handled separately — see deployment notes below.

## Next Steps

1. Update `.gitignore` to allow `data/deploy/` to be committed
2. Update `api/config.py` to load from `data/deploy/` in production
3. Handle model files via Railway environment or cloud storage
4. Deploy to Railway